In [31]:
import torch
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
import torch.nn as nn
import numpy as np
from tqdm import tqdm,trange


In [32]:
import torch

def patchify(images, patch_h, patch_w):
    n, c, h, w = images.shape
    assert h % patch_h == 0, f"Image height {h} not divisible by patch height {patch_h}"
    assert w % patch_w == 0, f"Image width {w} not divisible by patch width {patch_w}"
    
    num_patches_h = h // patch_h
    num_patches_w = w // patch_w
    
    patches = images.reshape(n, c, num_patches_h, patch_h, num_patches_w, patch_w)
    
    patches = patches.permute(0, 2, 4, 1, 3, 5)
    

    patches = patches.reshape(n, num_patches_h * num_patches_w, -1)
    
    return patches


In [33]:
import torch
import math

def get_positional_embedding(sequence_length, d):

    positions = torch.arange(sequence_length, dtype=torch.float32).unsqueeze(1)
    
    div_term = torch.exp(torch.arange(0, d, 2, dtype=torch.float32) * -(math.log(10000.0) / d))
    
    results = torch.zeros(sequence_length, d)
    
    results[:, 0::2] = torch.sin(positions * div_term)
    
    results[:, 1::2] = torch.cos(positions * div_term)
    
    return results

In [34]:
class MyMsa(nn.Module):
    def __init__(self,d,n_heads=2):
        super(MyMsa,self).__init__()
        self.d=d
        self.n_heads=n_heads

        assert d%n_heads==0

        d_head=int(d/n_heads)
        self.q_mappings=nn.ModuleList([nn.Linear(d_head,d_head) for _ in range(self.n_heads)])
        self.k_mappings=nn.ModuleList([nn.Linear(d_head,d_head) for _ in range(self.n_heads)])
        self.v_mappings=nn.ModuleList([nn.Linear(d_head,d_head) for _ in range(self.n_heads)])
        self.d_head=d_head
        self.softmax=nn.Softmax(dim=-1)
        self.out_proj=nn.Linear(d,d)

    def forward(self,sequences):
        result=[]
        for sequence in sequences:
            seq_result=[]
            for head in range(self.n_heads):
                q_mapping=self.q_mappings[head]
                k_mapping=self.k_mappings[head]
                v_mapping=self.v_mappings[head]

                seq=sequence[:,head*self.d_head:(head+1)*self.d_head]
                q,k,v=q_mapping(seq),k_mapping(seq),v_mapping(seq)

                attention=self.softmax(q@k.T/(self.d_head**2))
                seq_result.append(attention@v)

            result.append(torch.hstack(seq_result))

        return self.out_proj(torch.cat([torch.unsqueeze(r,dim=0) for r in result]))
                
        

In [35]:
class MyViTBlock(nn.Module):
    def __init__(self, hidden_d, n_heads, mlp_ratio=4):
        super(MyViTBlock, self).__init__()
        self.hidden_d = hidden_d
        self.n_heads = n_heads

        self.norm1 = nn.LayerNorm(hidden_d)
        self.mhsa = MyMsa(hidden_d, n_heads)
        self.norm2 = nn.LayerNorm(hidden_d)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_d, mlp_ratio * hidden_d),
            nn.GELU(),
            nn.Linear(mlp_ratio * hidden_d, hidden_d)
        )

    def forward(self, x):
        out = x + self.mhsa(self.norm1(x))
        out = out + self.mlp(self.norm2(out))
        return out

In [36]:
class MyVit(nn.Module):
    def __init__(self, chw=(3,224,224), n_patches=14, hidden_d=768, n_blocks=12, n_heads=12, out_d=10):
        super(MyVit,self).__init__()
        self.chw=chw
        self.n_patches=n_patches
        self.n_blocks=n_blocks
        self.n_heads=n_heads
        self.hidden_d=hidden_d
        self.out_d=out_d
        
        # Calculate physical pixel size of the patch
        self.patch_h = int(chw[1] / n_patches)
        self.patch_w = int(chw[2] / n_patches)

        assert chw[1] % n_patches == 0
        assert chw[2] % n_patches == 0 
        
        # 1. Linear Mapper (Calculates exactly 768)
        self.input_d= int(chw[0] * self.patch_h * self.patch_w)      
        self.linear_mapper=nn.Linear(self.input_d, self.hidden_d)

        # 2. Class Token
        self.class_token=nn.Parameter(torch.rand(1, self.hidden_d))

        # 3. Positional Embedding (Fixed UserWarning)
        pos_data = get_positional_embedding(self.n_patches**2 + 1, self.hidden_d)
        self.pos_embedding = nn.Parameter(pos_data.clone().detach())

        # 4. Transformer Blocks
        self.blocks=nn.ModuleList([MyViTBlock(hidden_d, n_heads) for _ in range(n_blocks)])

        # 5. MLP (Not used for DPT, but left here to prevent breaking old code)
        self.mlp=nn.Sequential(
            nn.Linear(self.hidden_d, self.out_d),
            nn.Softmax(dim=-1)
        )
    
    def forward(self, images):
        n, c, h, w = images.shape
        
        patches = patchify(images, self.patch_h, self.patch_w)
        
        tokens = self.linear_mapper(patches)
        
        # Add class token
        tokens = torch.stack([torch.vstack((self.class_token, tokens[i])) for i in range(len(tokens))])
        
        pos_embed = self.pos_embedding.repeat(n, 1, 1)
        out = tokens + pos_embed

        extracted_tokens = []
        for i, block in enumerate(self.blocks):
            out = block(out)
            if i in [2, 5, 8, 11]: 
                extracted_tokens.append(out)
                
        return extracted_tokens

In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ReassembleBlock(nn.Module):
    def __init__(self, in_channels, out_channels=256, spatial_size=14, scale_factor=1):
        super(ReassembleBlock, self).__init__()
        self.spatial_size = spatial_size
        
        self.project = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        
        if scale_factor > 1:
            self.resample = nn.ConvTranspose2d(
                out_channels, out_channels, 
                kernel_size=int(scale_factor*2), 
                stride=int(scale_factor), 
                padding=int(scale_factor//2)
            )
        elif scale_factor < 1:
            stride = int(1 / scale_factor)
            self.resample = nn.Conv2d(
                out_channels, out_channels, 
                kernel_size=3, stride=stride, padding=1
            )
        else:
            self.resample = nn.Identity()

    def forward(self, tokens):
        batch_size = tokens.shape[0]
        
        cls_token = tokens[:, 0:1, :]
        spatial_tokens = tokens[:, 1:, :]
        spatial_tokens = spatial_tokens + cls_token 
        
        grid = spatial_tokens.reshape(batch_size, self.spatial_size, self.spatial_size, -1)
        grid = grid.permute(0, 3, 1, 2) 
        
        out = self.project(grid)
        out = self.resample(out)
        
        return out

In [38]:
class ResidualConvUnit(nn.Module):
    def __init__(self, features):
        super(ResidualConvUnit, self).__init__()
        self.conv1 = nn.Conv2d(features, features, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(features, features, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.relu(x)
        out = self.conv1(out)
        out = self.relu(out)
        out = self.conv2(out)
        return out + x 

class FeatureFusionBlock(nn.Module):
    def __init__(self, features=256):
        super(FeatureFusionBlock, self).__init__()
        self.resConfUnit1 = ResidualConvUnit(features)
        self.resConfUnit2 = ResidualConvUnit(features)
        
        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.project = nn.Conv2d(features, features, kernel_size=3, padding=1)

    def forward(self, x, previous_stage_output=None):
        out = self.resConfUnit1(x)
        
        if previous_stage_output is not None:
            out = out + previous_stage_output
            
        out = self.resConfUnit2(out)
        out = self.upsample(out)
        out = self.project(out)
        
        return out

In [39]:
class DPT(nn.Module):
    def __init__(self, vit_encoder, embed_dim=768, features=256, spatial_size=14):
        super(DPT, self).__init__()
        self.encoder = vit_encoder
        
        
        self.reassemble_4 = ReassembleBlock(embed_dim, features, spatial_size, scale_factor=4)
        self.reassemble_8 = ReassembleBlock(embed_dim, features, spatial_size, scale_factor=2)
        self.reassemble_16 = ReassembleBlock(embed_dim, features, spatial_size, scale_factor=1)
        self.reassemble_32 = ReassembleBlock(embed_dim, features, spatial_size, scale_factor=0.5)
        
        self.fusion_32 = FeatureFusionBlock(features)
        self.fusion_16 = FeatureFusionBlock(features)
        self.fusion_8 = FeatureFusionBlock(features)
        self.fusion_4 = FeatureFusionBlock(features)
        
        self.head = nn.Sequential(
            nn.Conv2d(features, features // 2, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
            nn.Conv2d(features // 2, 1, kernel_size=1),
            nn.ReLU() 
        )

    def forward(self, x):
        layer_tokens = self.encoder(x)
        t3, t6, t9, t12 = layer_tokens 
        
        f4 = self.reassemble_4(t3)
        f8 = self.reassemble_8(t6)
        f16 = self.reassemble_16(t9)
        f32 = self.reassemble_32(t12)
        
        out = self.fusion_32(f32)
        out = self.fusion_16(f16, previous_stage_output=out)
        out = self.fusion_8(f8, previous_stage_output=out)
        out = self.fusion_4(f4, previous_stage_output=out)
        
        depth_map = self.head(out)
        
        return depth_map

In [40]:
import torch

def check_dpt_dimensions():
    # 1. Setup CUDA Device safely
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Using device: {device}\n")

    # 2. Define standard AR Face Try-On parameters
    batch_size = 2
    channels = 3
    height = 224
    width = 224
    n_patches = 14     # 14x14 grid = 196 patches (16x16 pixels each)
    embed_dim = 768    # Standard ViT-Base hidden dimension
    
    print("--- 1. Initializing Model ---")
    # 3. Build the ViT Encoder (Must have 12 blocks for the DPT hooks!)
    encoder = MyVit(
        chw=(channels, height, width), 
        n_patches=n_patches, 
        hidden_d=embed_dim, 
        n_blocks=12, 
        n_heads=12
    )
    
    # 4. Build the DPT Wrapper
    dpt_model = DPT(
        vit_encoder=encoder, 
        embed_dim=embed_dim, 
        features=256, 
        spatial_size=n_patches # 14
    ).to(device)
    print("✅ Model successfully built and moved to VRAM!\n")

    # 5. Create Dummy Input (e.g., a batch of 2 blank 224x224 RGB images)
    dummy_input = torch.randn(batch_size, channels, height, width).to(device)
    print(f"📷 Input Image Shape: {dummy_input.shape} -> [Batch, Channels, Height, Width]")

    print("\n--- 2. Checking ViT Extraction ---")
    # 6. Test just the ViT part to ensure tokens are extracted correctly
    layer_tokens = dpt_model.encoder(dummy_input)
    for i, tokens in enumerate(layer_tokens):
        # Should be [2, 197, 768] (Batch=2, 196 patches + 1 CLS token, 768 features)
        print(f"🔍 Extracted Layer {i+1} Shape: {tokens.shape}")

    print("\n--- 3. Checking Full DPT Pipeline ---")
    # 7. Test the full forward pass through Reassemble and Fusion
    try:
        depth_map = dpt_model(dummy_input)
        print(f"🎉 SUCCESS! Final Depth Map Shape: {depth_map.shape}")
        
        # Sanity Check: Ensure the output size matches the input size perfectly
        if depth_map.shape[-2:] == dummy_input.shape[-2:]:
            print("✅ Dimensions match! The network is ready for training.")
        else:
            print("❌ Dimension mismatch warning!")
            
    except Exception as e:
        print(f"❌ FORWARD PASS FAILED: {e}")

# Run the test
check_dpt_dimensions()

🚀 Using device: cuda

--- 1. Initializing Model ---
✅ Model successfully built and moved to VRAM!

📷 Input Image Shape: torch.Size([2, 3, 224, 224]) -> [Batch, Channels, Height, Width]

--- 2. Checking ViT Extraction ---
🔍 Extracted Layer 1 Shape: torch.Size([2, 197, 768])
🔍 Extracted Layer 2 Shape: torch.Size([2, 197, 768])
🔍 Extracted Layer 3 Shape: torch.Size([2, 197, 768])
🔍 Extracted Layer 4 Shape: torch.Size([2, 197, 768])

--- 3. Checking Full DPT Pipeline ---
🎉 SUCCESS! Final Depth Map Shape: torch.Size([2, 1, 224, 224])
✅ Dimensions match! The network is ready for training.


In [41]:
# def main():
#     transform=transforms.ToTensor()
#     train_data=datasets.mnist.MNIST(root="./../datasets",train=True,download=True,transform=transform)
#     test_data=datasets.mnist.MNIST(root="./../datasets",train=False,download=True,transform=transform)

#     train_loader=DataLoader(train_data,shuffle=True,batch_size=128)
#     test_loader=DataLoader(test_data,shuffle=False,batch_size=128)
#     device=torch.device("cuda")
#     print(f"using device {device}")

#     model=MyVit((1,28,28),n_patches=7,n_blocks=2,hidden_d=8,n_heads=2,out_d=10).to(device)
#     n_epochs=5
#     lr=0.005

#     optimizer=Adam(model.parameters(),lr=lr)
#     criterion=CrossEntropyLoss()

#     for epoch in trange(n_epochs,desc="training"):
#         train_loss=0.0
#         for batch in train_loader:
#             x,y=batch
#             x,y=x.to(device),y.to(device)
#             y_hat=model(x)
#             loss=criterion(y_hat,y)
#             train_loss+=loss.detach().cpu().item() /len(train_loader)

#             optimizer.zero_grad()
#             loss.backward()
#             optimizer.step()
#         print(f"epoch {epoch+1}/{n_epochs} loss:{train_loss:.2f}")

#     with torch.no_grad():
#         correct,total=0,0
#         test_loss=0.0
#         for batch in tqdm(test_loader,desc="testing"):
#             x,y=batch
#             x,y=x.to(device),y.to(device)
#             y_hat=model(x)
#             loss=criterion(y_hat,y)
#             test_loss+=loss.detach().cpu().item() /len(test_loader)
#             correct+=torch.sum(torch.argmax(y_hat,dim=1)==y).detach().cpu().item()
#             total+=len(x)

#         print(f"test loss :{test_loss:.2f}")
#         print(f"test accuracy: {correct/total *100:.2f}")

    
            

# main()